# OpenClaw

## Setup

The below steps apply to the Linux operating system. Refer to the [documentation](https://docs.openclaw.ai/start/getting-started) for other operating systems.

#### Install OpenClaw

In [ ]:
!curl -fsSL https://openclaw.ai/install.sh | bash

#### Install the openclaw service.  
Note that this skips the onboarding, we will configure the settings manually.

In [ ]:
!openclaw gateway --install-daemon

#### Verify gateway is running

In [ ]:
!openclaw gateway status

## Configure OpenClaw

#### Backup the default openclaw config.

In [ ]:
!cp $HOME/.openclaw/openclaw.json $HOME/.openclaw/openclaw.json.default

#### Create music store agent

In [ ]:
!openclaw agents add music-store-agent

#### Specify model attributes

We will use the [nemotron 3 super 120b](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b) model as the LLM backend for OpenClaw.  
Although this model supports up to 1M tokens, we've restricted the context window to 100k tokens.

Replace `<your_api_key>` with your api key in the json below.  
Replace `models` in `$HOME/.openclaw/openclaw.json` with the completed json

```json
"models": {
    "mode": "merge",
    "providers": {
    "nvidia-nim": {
        "baseUrl": "https://integrate.api.nvidia.com/v1",
        "apiKey": "<your_api_key>",
        "api": "openai-completions",
        "models": [
            {
                "id": "nvidia/nemotron-3-super-120b-a12b",
                "name": "nvidia/nemotron-3-super-120b-a12b (Custom Provider)",
                "reasoning": false,
                "input": [
                    "text"
                ],
                "cost": {
                "input": 0,
                "output": 0,
                "cacheRead": 0,
                "cacheWrite": 0
                },
                "contextWindow": 100000,
                "maxTokens": 4096
            }
        ]
    }
    }
}
```

#### Specify agents attributes

Replace `<your_home_directory>` with your actual home directory in the json below.   
Replace `agents` in `$HOME/.openclaw/openclaw.json` with the completed json.

```json
"agents": {
    "defaults": {
      "model": {
        "primary": "nvidia-nim/nvidia/nemotron-3-super-120b-a12b"
      },
      "models": {
        "nvidia-nim/nvidia/nemotron-3-super-120b-a12b": {}
      },
      "workspace": "<your_home_directory>/.openclaw/workspace"
    },
    "list": [
      {
        "id": "main"
      },
      {
        "id": "music-store-agent",
        "name": "music-store-agent",
        "workspace": "<your_home_directory>/.openclaw/workspace-music-store-agent",
        "agentDir": "<your_home_directory>/.openclaw/agents/music-store-agent/agent",
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "default": true
      }
    ]
}
```

#### Define the music store agent

Define the behaviour of the agent in `SOUL.md`. 

In [ ]:
!cat > $HOME/.openclaw/workspace-music-store-agent/SOUL.md << 'EOF'
# Music Store Agent

You are a music store agent. Do not invent anything. All data comes from skills.

## Allowed Skills
- music-store-assistant
EOF

#### Create agent skill

Create the skills folder in the music store agent.  

In [ ]:
!mkdir -p $HOME/.openclaw/workspace-music-store-agent/skills/music-store-assistant

Add contents to `SKILL.md`.

The instruction directs the agent to issue a `curl` command that invokes the workflow by hitting the endpoint of the NAT webserver.

In [ ]:
!cat > $HOME/.openclaw/workspace-music-store-agent/skills/music-store-assistant << 'EOF'
---
name: music-store-assistant
description: lookup information on tracks,artists,abums and process refunds.
---

### Query the music store assistant

Replace <query> with question from user/agent. Do not remove any information.

```bash
curl -X POST http://localhost:8001/generate \
  -H "Content-Type: application/json" \
  -d $'{
    "input_message": <query>
  }'
EOF

#### Specify host binaries allowed to be run by the agent

Replace `agents` in `$HOME/.openclaw/exec-approvals.json` with the below json.

```json
"agents": {
    "music-store-agent": {
      "security": "allowlist",
      "ask": "on-miss",
      "allowlist": [
        {
          "pattern": "/usr/bin/curl"
        }
      ]
    }
  }
```

#### Restart openclaw gateway to effect the changes made above

In [ ]:
!openclaw gateway restart

#### Serve the completed workflow from [assignment 4 - nemo agent toolkit](../../challenge/nemo_agent_toolkit/) as a web server

In [ ]:
!nat serve --port 8001 --config_file ../../challenge/nemo_agent_toolkit/workflaw.yaml

#### Test the web endpoint

In [ ]:
!curl -X POST http://localhost:8001/generate \
  -H "Content-Type: application/json" \
  -d $'{
    "input_message": "my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice IDs related to the artist Led Zeppelin"
  }'

#### Run OpenClaw TUI

In [ ]:
!openclaw tui

#### Query the music store agent

Input the following to the openclaw tui

`my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice IDs related to the artist Led Zeppelin`

Likely output:

```
The invoice IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**.
```

#### View session traces

List files in the sessions folder of the agent

In [ ]:
!ls $HOME/.openclaw/agents/music-store-agent/sessions/

Likely output:

```
57e6a68e-3873-43ca-a851-6b085315f4c1.jsonl  sessions.json
```

Replace `<sessionId>` with 1 of the jsonl files found in the directory

In [ ]:
!cat $HOME/.openclaw/agents/music-store-agent/sessions/<sessionId>.jsonl

Like output:

```json
{"role":"user","content":[{"type":"text","text":"Sender (untrusted metadata):\n```json\n{\n  \"label\": \"openclaw-tui\",\n  \"id\": \"openclaw-tui\",\n  \"name\": \"openclaw-tui\",\n  \"username\": \"openclaw-tui\"\n}\n```\n\n[Fri 2026-04-10 07:39 UTC] my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice  IDs related to the artist Led Zeppelin"}]}

{"role":"assistant","content":[{"type":"thinking","thinking":"We need to get the invoice IDs related to artist Led Zeppelin. First, I should check the available skills and see what tools I have. Looking at the skills, there's a music-store-assistant skill. I should read that to understand how to use it. Let me read the SKILL.md for music-store-assistant.\n\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-983ae2ee723dbbd6","name":"read","arguments":{"path":"/home/ubuntu/.openclaw/workspace-music-store-agent/skills/music-store-assistant/SKILL.md"}}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","stopReason":"toolUse"}

{"role":"toolResult","toolCallId":"chatcmpl-tool-983ae2ee723dbbd6","toolName":"read","content":[{"type":"text","text":"---\nname: music-store-assistant\ndescription: lookup information on tracks,artists,abums and process refunds.\n---\n\n### Query the music store assistant\n\nReplace <query> with exact statement from user/agent. Do not remove any information.\n\n```bash\ncurl -X POST http://localhost:8001/generate \\\n  -H \"Content-Type: application/json\" \\\n  -d $'{\n    \"input_message\": <query>\n  }'\n```\n"}]}

{"role":"assistant","content":[{"type":"thinking","thinking":"We need to call the music store assistant skill to get invoice IDs related to artist Led Zeppelin. According to the skill description, I need to use curl to POST to http://localhost:8001/generate with input_message containing the user's query exactly.\n\nUser's query: \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice IDs related to the artist Led Zeppelin\"\n\nI should include that exact statement as input_message. Let me call the tool.\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-bdef75da29fa3b69","name":"exec","arguments":{"command":"curl -X POST http://localhost:8001/generate -H \"Content-Type: application/json\" -d $'{\"input_message\": \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice IDs related to the artist Led Zeppelin\"}'"}}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","stopReason":"toolUse"}

{"role":"toolResult","toolCallId":"chatcmpl-tool-bdef75da29fa3b69","toolName":"exec","content":[{"type":"text","text":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   271  100   124  100   147  52698  62473 --:--:-- --:--:-- --:--:--  132k\n{\"detail\":[{\"type\":\"json_invalid\",\"loc\":[\"body\",0],\"msg\":\"JSON decode error\",\"input\":{},\"ctx\":{\"error\":\"Expecting value\"}}]}"}],"details":{"status":"completed","exitCode":0,"durationMs":30,"aggregated":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   271  100   124  100   147  52698  62473 --:--:-- --:--:-- --:--:--  132k\n{\"detail\":[{\"type\":\"json_invalid\",\"loc\":[\"body\",0],\"msg\":\"JSON decode error\",\"input\":{},\"ctx\":{\"error\":\"Expecting value\"}}]}","cwd":"/home/ubuntu/.openclaw/workspace-music-store-agent"}}

{"role":"assistant","content":[{"type":"thinking","thinking":"The user is asking for invoice IDs related to the artist Led Zeppelin. I need to call the music store assistant skill. The skill description says it can lookup information on tracks, artists, albums and process refunds. The user wants invoice IDs related to Led Zeppelin. I should use the skill's query format. The skill.md shows how to call it: a curl POST to http://localhost:8001/generate with JSON input_message. I need to include the user's message exactly. However, the previous attempt failed because I didn't properly format the JSON. The input_message should be a string containing the user's request. I need to escape it properly. Let me try again with correct JSON formatting. I'll use the exec tool with proper quoting. The user said: \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice IDs related to the artist Led Zeppelin\". I'll include that as the input_message. Let's do it.\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-8fb781cff172d362","name":"exec","arguments":{"command":"curl -X POST http://localhost:8001/generate -H \"Content-Type: application/json\" -d '{\"input_message\": \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice IDs related to the artist Led Zeppelin\"}'"}}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","stopReason":"toolUse"}

{"role":"toolResult","toolCallId":"chatcmpl-tool-8fb781cff172d362","toolName":"exec","content":[{"type":"text","text":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   146    0     0  100   146      0    121  0:00:01  0:00:01 --:--:--   121\r100   146    0     0  100   146      0     66  0:00:02  0:00:02 --:--:--    66\r100   146    0     0  100   146      0     45  0:00:03  0:00:03 --:--:--    45\r100   146    0     0  100   146      0     34  0:00:04  0:00:04 --:--:--    34\r 56   257    0     0  100   146      0     29  0:00:05  0:00:04  0:00:01    29\r100   257  100   111  100   146     22     29  0:00:05  0:00:04  0:00:01    29\n{\"value\":\"The invoice IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**.\"}"}],"details":{"status":"completed","exitCode":0,"durationMs":4977,"aggregated":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   146    0     0  100   146      0    121  0:00:01  0:00:01 --:--:--   121\r100   146    0     0  100   146      0     66  0:00:02  0:00:02 --:--:--    66\r100   146    0     0  100   146      0     45  0:00:03  0:00:03 --:--:--    45\r100   146    0     0  100   146      0     34  0:00:04  0:00:04 --:--:--    34\r 56   257    0     0  100   146      0     29  0:00:05  0:00:04  0:00:01    29\r100   257  100   111  100   146     22     29  0:00:05  0:00:04  0:00:01    29\n{\"value\":\"The invoice IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**.\"}","cwd":"/home/ubuntu/.openclaw/workspace-music-store-agent"}}

{"role":"assistant","content":[{"type":"thinking","thinking":"We have received the response from the music store assistant: the invoice IDs related to the artist Led Zeppelin for Aaron Mitchell are 267 and 268. I should present this answer to the user. Let's do that.\n","thinkingSignature":"reasoning_content"},{"type":"text","text":"The invoice IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**."}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","stopReason":"stop"}
```